# N026 · 同向双指针与稳定压缩

**目标：** 用不同移动速度维护有效前缀和相对顺序。

**先修：** N017, N025。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 读写指针；重复计数；最多保留k次；稳定性。

**配套讲解来源：** [同名逐章意见](../../comment/026_same_direction_fast_slow.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决"原地删重复、但删多狠可调"的问题：给你一个**已按升序排好**的数组，要求原地修改，让每个值至多保留 k 次，返回有效长度；k=1 就是普通去重（每个值只留一份）。

举个具体到数字的例子：输入 `[1,1,1,2,2,3]`、k=2 → 输出有效长度 5，数组前 5 格变成 `[1,1,2,2,3]` → 为什么？1 出现 3 次超过上限 2，砍掉一个；2 出现 2 次恰好达标，全留；3 出现 1 次，照留。数组物理长度不变（还是 6），“运行示例”部分残留的旧数据不算答案——调用方按返回的有效长度取前缀，这和 N024 的 compress 是同一份契约。

同一个输入换个 k 答案就变：k=1 得 `[1,2,3]`（长 3），k=3 得 `[1,1,1,2,2,3]`（长 6，一个都不用砍）。本章把"删到剩几个"参数化成一个函数，普通去重只是 k=1 的特例。

问题里还藏着两个隐性要求：一是**原地**（只许常数个额外变量），二是**稳定**（留下的元素保持原有相对顺序）。排序数组里相同值天然相邻，这是整个算法能一步判断"超没超"的根基。

## 2. 基础知识：先读懂这些词

- **同向双指针（快慢指针）**：两根指针都从左往右走、方向相同但速度不同——read（快）负责扫遍全数组，write（慢）只在"决定保留"时前进一格。与 N025 的相向双指针相反，这里不是两端夹中间，而是同赛道一前一后。
- **读写指针（read / write）**：read 读原数据、write 写结果。因为写出的永远不比读掉的多（我们只在删东西），write 永远落后或贴着 read，所以原地覆盖不会毁掉还没读的内容。
- **有效前缀（valid prefix）**：数组开头 0 到 write−1 这一段，是"已经敲定保留"的最终答案。函数运行的全过程中，有效前缀里每个值都保持"至多 k 个"——这是算法从头到尾维持的核心性质（术语叫不变量：每轮循环后都为真的事实）。
- **最多保留 k 次（keep at most k）**：每个不同的值在答案里出现次数的上限。k=1 是严格去重，k=2 是 LeetCode 80 题的要求，k=0 是全删。
- **重复计数的关键比较（`value != nums[write-k]`）**：判断"当前值还能不能写"的唯一依据。`nums[write-k]` 是已写前缀的倒数第 k 个元素——如果它等于当前值，说明前缀尾部已经攒了 k 个和当前值相同的元素，再写就是第 k+1 个，超限；否则说明尾部同值不足 k 个，写一个安全。
- **有序前提（sortedness）**：算法依赖"相同的值在数组里连成一片"。只有这样，"前缀尾部连着 k 个同值"才能由"倒数第 k 个等于当前值"一步推出——因为有序前缀里同值必然相邻，中间不可能夹着别的数。
- **稳定性（stability）**：保留的元素之间的先后关系与原数组一致。本章按读到的顺序逐个写入，天然稳定。
- **长度契约**：函数返回有效长度 k，答案看 `nums[:k]`，之后是待复用的空位或旧数据，不许读。与 N024 的 compress 一致。

## 3. 思路推导：从小例子开始

- **Step 1：先挡住不合法的上限。** k≤0 意味着什么都不许留，直接返回 0（同时也是给后面的 `nums[write-k]` 挡住负下标）。
- **Step 2：read 从头扫到尾，write 从 0 候命。** 每读到一个值，问一个问题："把它写进有效前缀，会不会让某个值的个数超过 k？"
- **Step 3：用一条规则回答。** 前缀还不足 k 长（`write<k`），随便写，肯定不超；否则看前缀倒数第 k 个元素 `nums[write-k]`：它不等于当前值就写，等于就不写。
- **Step 4：read 走完，write 就是有效长度。**

**为什么这条规则就够了？** 分两种情况想。第一种：当前值在前缀尾部已经连排了 k 个。因为前缀是从有序数组里按顺序写出来的，它自己也有序；有序的数组里相等的值相邻，所以"倒数第 k 个等于当前值"就意味着最后 k 个全是当前值——再写就是第 k+1 个，必须拒。第二种：倒数第 k 个不等于当前值。前缀有序，说明从倒数第 k 个往前都不小于……换个直接的说法：当前值即使与尾部某些元素相同，同值的连排长度也到不了 k（否则倒数第 k 个就会被卷进去等于当前值），所以写一个绝不会超限。一条比较同时防住"超限"和"误杀"，去重（k=1）与保留 k 次共用同一套逻辑——这正是 “思路推导”部分 说的"让一个规则同时解决去重与有限次数保留"。

**手算演示：`keep_at_most_k([1,1,1,2,2,3], k=2)` 一格一格走**

| read | 当前值 | write | 检查过程 | 写入？ | 有效前缀（写完后） |
|---|---|---|---|---|---|
| 0 | 1 | 0 | write=0 < k=2，直接放行 | 是 → write=1 | [1] |
| 1 | 1 | 1 | write=1 < 2，放行 | 是 → write=2 | [1,1] |
| 2 | 1 | 2 | write=2 不小于 2；nums[0]=1 等于当前值 | **否**（已是第 3 个 1） | [1,1] |
| 3 | 2 | 2 | nums[0]=1 ≠ 2，放行 | 是 → write=3 | [1,1,2] |
| 4 | 2 | 3 | nums[1]=1 ≠ 2，放行 | 是 → write=4 | [1,1,2,2] |
| 5 | 3 | 4 | nums[2]=2 ≠ 3，放行 | 是 → write=5 | [1,1,2,2,3] |

返回 5，前缀 `[1,1,2,2,3]`。注意 read=2 那一行是全场唯一的拒绝，它精确砍掉了第三个 1。“运行示例”部分 的表格正是用这同一个数组对 k=0,1,2,3 各跑一遍：k=0 得长度 0、k=1 得 `[1,2,3]`、k=2 得 `[1,1,2,2,3]`、k=3 得 `[1,1,1,2,2,3]`（一次都不砍）。

**手算演示（对照组）：同一个数组、k=1**

| read | 当前值 | write | 检查过程 | 写入？ | 有效前缀 |
|---|---|---|---|---|---|
| 0 | 1 | 0 | write=0 < k=1，放行 | 是 → write=1 | [1] |
| 1 | 1 | 1 | nums[write−1]=nums[0]=1 等于当前值 | **否** | [1] |
| 2 | 1 | 1 | nums[0]=1 等于当前值 | **否** | [1] |
| 3 | 2 | 1 | nums[0]=1 ≠ 2，放行 | 是 → write=2 | [1,2] |
| 4 | 2 | 2 | nums[1]=1 ≠ 2，放行 | 是 → write=3 | [1,2,3] |
| 5 | 3 | 3 | nums[2]=2 ≠ 3，放行 | 是 → write=3+1 | [1,2,3] |

对比 k=2 与 k=1 两张表你会发现有趣的一点：read=3、4 处的 2 在 k=1 表里比较的对象是 `nums[write-1]`（前缀最后一个），在 k=2 表里是 `nums[write-2]`（倒数第二个）——同一套代码、同一次扫描，仅比较位置不同就实现了两种保留策略。

**k=1 的特例视角：** `remove_duplicates([1,1,2])` 调用 `keep_at_most_k(a,1)`。read=0：write=0<1 放行写 1；read=1：nums[write−1]=nums[0]=1 等于当前值，拒；read=2：nums[0]=1 ≠ 2，写。得长度 2、前缀 `[1,2]`。k=1 时"倒数第 1 个"就是前缀最后一个元素，规则退化为"和上一个写下的不同才写"——这正是经典的有序去重写法。

## 4. 核心代码：remove_duplicates

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def remove_duplicates(nums):
    return keep_at_most_k(nums, 1)
```

### 逐段读懂代码

### 函数 1：`keep_at_most_k`（最多保留 k 次，本章核心）

先贴 notebook “分段实现”部分 里的真实代码：

```python
def keep_at_most_k(nums, k):
    if k <= 0:
        return 0
    write = 0
    for read in range(len(nums)):
        value = nums[read]
        if write < k or value != nums[write - k]:
            nums[write] = value
            write += 1
    return write
```

- **第 2 行 `if k<=0: return 0`**：上限闸门。k=0 表示"全部删除"，有效长度就是 0；k 为负数更是无意义的输入。这一行还兼任护盾：没有它，后面 `nums[write-k]` 在 k=0 时会取到 `nums[write]`（应该是 `nums[write-0]`，语义完全错了），k 为负时更是直接算出越界下标。
- **第 3 行 `write=0`**：写指针从 0 出发，此刻有效前缀为空。
- **第 4–5 行 `for read in range(len(nums)): value=nums[read]`**：read 是快指针，把数组从头到尾扫一遍；当前值缓存进 value，让下一行的条件好读。
- **第 6 行 `if write<k or value!=nums[write-k]:`**：全章的灵魂判断，两个条件用 or 连接——**满足任意一个就写**。前半 `write<k`：前缀还不足 k 个元素，无论什么值写进去都不可能让某个值超过 k 个（每个值最多占 k 个名额，总共还没有 k 个元素呢）；后半 `value!=nums[write-k]`：前缀已经够长了，那就看倒数第 k 个写下的元素——不等就说明前缀尾部的同值连排不足 k 个，写一个安全；相等说明已经攒满 k 个，拒写。注意比较对象是**已写出的前缀**（随 write 动态变化），不是原数组的固定位置——“思路推导”部分 特意强调这一点。
- **第 6 行后半 `nums[write]=value; write+=1`**：判定通过，当前值写进 write 位置，write 前进一格。分号连写是紧凑风格，等价于两行。
- **第 8 行 `return write`**：循环结束，write 走过几格答案就有几个元素，返回有效长度。

**为什么 write 不会踩坏没读的数据？** 每轮循环里 read 前进 1 格，write 至多前进 1 格，且 write 的初值不大于 read 的初值，所以任何时刻 write ≤ read+1。更直接地看：写入发生的条件本身就保证写出的数量不超过读入的数量，write 永远追不上 read 要读的下一个位置，原地覆盖安全。拿 `[1,1,1,2,2,3]`、k=2 的推演数一数：全程读 6 个、写 5 个，write 始终比 read 慢半拍或齐平，从未越过读的位置。

**同一骨架在三章之间的对照**：N024 的 compress、本章的 keep_at_most_k、以及 N025 的 two_sum_sorted，指针结构可以排成一条演化线——compress 与本章都是"同向 read/write 双指针、按条件写入"，区别在写入条件一个是"游程压缩规则"、一个是"至多 k 次"；N025 则是"相向双指针、按大小关系扔端点"。读代码时先认骨架再看条件，三个函数就是一套语言的三个句子。

### 函数 2：`remove_duplicates`（严格去重，薄封装）

```python
def remove_duplicates(nums):
    return keep_at_most_k(nums, 1)
```

只有一行：把 k 固定为 1 转交给核心函数。它存在的意义是对齐 LeetCode 26 题的函数名——"每个值只留一份"就是"至多保留 1 次"。你也由此看到两个 LeetCode 题（26 和 80）在本章框架里只是同一个参数 k 的两个取值。

## 5. 分段实现：按顺序运行

**本章接口：** `remove_duplicates(nums)`、`keep_at_most_k(nums, k)`

### keep_at_most_k

In [1]:
def keep_at_most_k(nums, k):
    if k <= 0:
        return 0
    write = 0
    for read in range(len(nums)):
        value = nums[read]
        if write < k or value != nums[write - k]:
            nums[write] = value
            write += 1
    return write

### remove_duplicates

In [2]:
def remove_duplicates(nums):
    return keep_at_most_k(nums, 1)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,1,1,2,2,3]; rows=[]
for k in [0,1,2,3]:
    b=a[:]; length=keep_at_most_k(b,k); rows.append((k,length,b[:length]))
show_table(['保留上限','有效长度','有效前缀'],rows)

保留上限,有效长度,有效前缀
0,0,[]
1,3,"[1, 2, 3]"
2,5,"[1, 1, 2, 2, 3]"
3,6,"[1, 1, 1, 2, 2, 3]"


## 7. 正确性、适用条件与复杂度

有序性确保相同值连续。当nums[write−k]等于当前值时，前缀尾部已经保留k个该值；否则再保留一个不会超限。

**代价：** O(n) 时间、O(1) 辅助空间；仅对有序输入成立。

### 展开理解

**为什么是对的？** 我们要确认三件事：留下的没超限、该留的没漏、顺序没乱。

先看"没超限"。整个运行过程中有一个始终成立的事实：有效前缀（0 到 write−1）里每个值至多出现 k 次。初始时前缀为空，事实成立；每一次写入前，规则要么走 `write<k`（前缀总长都不足 k，单值更不可能超），要么走 `value!=nums[write-k]`（有序前缀里若尾部已有 k 个当前值，则倒数第 k 个必然也是当前值——因为有序数组里相等值相邻，k 个同值必然连成一片占据尾部——比较就会失败；比较通过说明尾部同值不足 k 个）。所以每次写入后事实依然成立，循环结束、返回时的前缀自然满足"每值至多 k 个"。

再看"没漏、没乱"。read 把每个元素都过目了一遍，凡是写入不超限的元素全部被写入（规则只在"写了会超限"时才拒绝），所以砍掉的恰好是每个连排里超出 k 的部分，一个不多一个不少；又因为写入顺序与读取顺序一致，保留元素之间的相对次序和原数组相同——这就是"稳定"。

最后看前提。以上论证每一步都用了"输入有序"：有序保证了同值相邻，"倒数第 k 个等于当前值 ⇒ 尾部 k 个都是当前值"这步推理才成立。输入无序时同值被隔开，规则可能对一个"隔了半条街的旧值"做出错误判断，算法就会失灵——“正确性与复杂度”部分 的"仅对有序输入成立"说的就是这个。

**复杂度是多少？** 时间 O(n)：read 恰好走 n 格，每格做常数次比较和至多一次写入。n=10⁵ 时大约十万次判断，一瞬间完成。空间 O(1)：全程只有 read、write、value、k 这几个变量，没有开任何新数组。对比"用哈希表计数再重建"的思路（额外 O(n) 空间），本章做法把空间压到常数，代价只是要求输入有序。

**把三种做法放在同一把尺子上量**（以 n=10⁵、全是同一个值、k=2 为例）：哈希表版要先扫一遍建计数（10⁵ 次哈希更新）、再按序铺开（10⁵ 次写入）；排序+重扫版要先 O(n log n) 排序（约 170 万次比较）；本章版在已有序的前提下只做 10⁵ 次"一次比较加至多一次写入"。前提（输入已有序）越充分，算法就能省得越狠——这是本章和 N025 反复出现的主题。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条看：

- `a=[1,1,2]; k=remove_duplicates(a); assert a[:k]==[1,2]`：k=1 的正常值（也是 LeetCode 26 的样例）。测的是"连排超过 1 个的部分被砍、砍完前缀正确"，同时示范用 `a[:k]` 按有效长度验收。
- `a=[1,1,1,2,2,3]; k=keep_at_most_k(a,2); assert a[:k]==[1,1,2,2,3]`：k=2 的正常值（LeetCode 80 的样例）。三个 1 砍成一个、两个 2 刚好保留，测的是"恰好达标的连排不误杀"。
- `assert remove_duplicates([])==0`：空数组边界。循环一次不进、write 保持 0，测的是空输入安全返回。顺着这条再补两个值得自己口头过的边界：单元素数组 `[5]`——write=0<1 放行写入、返回 1，前缀 `[5]`；全部互异 `[1,2,3]`、k=2——每个值都只出现一次，谁也不会被砍，返回 3、前缀原样。这两个边界没进断言，但面试官常拿它们追问。

| 输入 | k | 返回 | 有效前缀 | 考验点 |
|---|---|---|---|---|
| `[5]` | 1 | 1 | `[5]` | 单元素放行 |
| `[1,2,3]` | 2 | 3 | `[1,2,3]` | 无重复时全保留 |
- **三重循环对拍**：`for n in range(7)` 枚举长度 0 到 6，`combinations_with_replacement([0,1,2],n)` 枚举该长度下所有**有序**数组（从 0,1,2 里可重复选取，天然升序，满足算法前提），`for limit in range(4)` 枚举上限 0 到 3。参考答案 `ref` 用另一种方式现算：`Counter(a)` 数出每个值的出现次数 c，每个值重复 `min(c,limit)` 次，按值排序铺开。然后断言 `b[:k]==ref`——等于拿一个"先数数再铺开"的朴素实现，和本章"一遍扫描边读边写"的实现，在几百种小输入上逐一对答案。这组对拍同时覆盖了空数组、全相同、全不同、上限为 0 等边界。

全部通过后打印 `N026: 所有本章断言通过`。

In [4]:
a = [1, 1, 2]

k = remove_duplicates(a)

assert a[:k] == [1, 2]

a = [1, 1, 1, 2, 2, 3]

k = keep_at_most_k(a, 2)

assert a[:k] == [1, 1, 2, 2, 3]

assert remove_duplicates([]) == 0

from itertools import combinations_with_replacement

from collections import Counter

for n in range(7):
    for a in combinations_with_replacement([0, 1, 2], n):
        for limit in range(4):
            b = list(a)
            k = keep_at_most_k(b, limit)
            ref = [x for x, c in sorted(Counter(a).items()) for _ in range(min(c, limit))]
            assert b[:k] == ref

print('N026: 所有本章断言通过')

N026: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 从最多一次推广到最多k次。

**练习 2：** 说明输入必须有序。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（从最多一次推广到最多 k 次）**：方向是先假装不知道本章代码，从 k=1 的经典去重出发自己改。提示：k=1 时你比较的是"前缀最后一个"（`nums[write-1]`），推广时问自己"比较对象应该往后挪几位"——答案是挪成倒数第 k 个 `nums[write-k]`，再加上"前缀不足 k 长时无条件放行"的前半个条件。手算示例直接用 `[1,1,1,2,2,3]`、k=2，把第三节那张表自己重画一遍，重点标出唯一被拒的 read=2。边界要覆盖 k=0（全删）、k 大于数组长度（全留）、空数组。
- **练习 2（说明输入必须有序）**：方向是找一个无序输入让规则给出错答案。提示：试 `[1,2,1]`、k=1——手算过程是 read=0 写 1，read=1 时 `nums[write-1]=1` 不等于 2、写 2，read=2 时 `nums[write-1]=2` 不等于 1、又写！结果前缀 `[1,2,1]`，1 出现了两次，违背"至多 1 次"。你要在表里指出哪一步"同值相邻"的假设被打破了（两个 1 中间隔了个 2），并总结：无序时比较 `nums[write-k]` 看到的可能是毫不相干的值，判断随之失效。验证可以用"先排序再调用"和"直接调用"两种方式跑同一个输入对比结果。

**两个高频踩坑点提前打个预防针：**

1. **比较对象写成 `nums[read-k]`**：这是最常见的手滑。比较必须基于**已写出的前缀**（`nums[write-k]`），不是原数组里 read 左边 k 格的元素——被拒绝的元素根本不该进前缀，用 read 位置往前数会把被删掉的值也算进去，判断随之错位。
2. **忘了 `write<k` 的放行条件**：只用 `value!=nums[write-k]` 的话，前缀不足 k 个元素时会出现负下标（write<k 时 write−k<0，Python 会从数组尾部取值，错误还不报错，极难发现）。前半个条件既是策略（总长不足 k 必然安全）也是护盾（挡住负下标）。

做完两题你应该能脱口而出本章的关键比较对象：`nums[write-k]`——"已写前缀的倒数第 k 个"，以及它为什么必须踩在有序性上才站得住。

做完两题你应该能脱口而出本章的关键比较对象：`nums[write-k]`——"已写前缀的倒数第 k 个"，以及它为什么必须踩在有序性上才站得住。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def keep_at_most_k(nums, k):
    if k <= 0:
        return 0
    write = 0
    for read in range(len(nums)):
        value = nums[read]
        if write < k or value != nums[write - k]:
            nums[write] = value
            write += 1
    return write

def remove_duplicates(nums):
    return keep_at_most_k(nums, 1)

# 示例与回归测试
a = [1, 1, 2]

k = remove_duplicates(a)

assert a[:k] == [1, 2]

a = [1, 1, 1, 2, 2, 3]

k = keep_at_most_k(a, 2)

assert a[:k] == [1, 1, 2, 2, 3]

assert remove_duplicates([]) == 0

from itertools import combinations_with_replacement

from collections import Counter

for n in range(7):
    for a in combinations_with_replacement([0, 1, 2], n):
        for limit in range(4):
            b = list(a)
            k = keep_at_most_k(b, limit)
            ref = [x for x, c in sorted(Counter(a).items()) for _ in range(min(c, limit))]
            assert b[:k] == ref

print('N026: 所有本章断言通过')

N026: 所有本章断言通过


## 11. 代表题与迁移

- **26. Remove Duplicates from Sorted Array（删除有序数组中的重复项）**：这题就是 `remove_duplicates` 本尊（k=1 特例），练的是读写指针骨架和"与前一个写下的值比较"的判断；返回有效长度、后缀不作数是它的高频坑。
- **80. Remove Duplicates from Sorted Array II（删除有序数组中的重复项 II）**：这题就是 `keep_at_most_k(nums,2)` 本尊，练的是把比较对象从倒数第 1 个推广到倒数第 k 个；两题共用本章同一个函数，差别只是参数。

两题都标为 canonical（与本章一一对应）。迁移方向：把"最多保留 k 次"里的比较基准换成别的谓词（比如"保留满足某条件的元素"），读写指针骨架照用；再往远走，Move Zeroes（283）这类"原地过滤"题也是同一个 read/write 模板的直接变体。

自测问题：如果 k 远大于数组长度，`write<k` 这个条件会一直放行吗？（会，且这是正确的——总长都不够 k，任何值都不可能超限。）

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。